# Predicting Stellar Class — Modeling

Kaggle Playground Series S6E6. Multiclass: `GALAXY` / `STAR` / `QSO`. Metric — **balanced accuracy**.

This notebook is **self-contained**: it reads `train.csv` / `test.csv` directly, optionally
appends the original **SDSS17** dataset (cleaned of `-9999` sentinels), rebuilds the engineered
features, trains a **LightGBM + CatBoost + XGBoost** ensemble with 5-fold `StratifiedKFold`,
**stacks** the OOF probabilities with a logistic-regression meta-model, tunes **per-class
decision weights** for balanced accuracy, and writes `submission.csv`.

Designed to run on Kaggle with 2x T4 GPUs (each booster uses GPU where available).
To use the external data: set `USE_EXTERNAL = True` and add the
[Stellar Classification Dataset SDSS17](https://www.kaggle.com/datasets/fedesoriano/stellar-classification-dataset-sdss17)
as a Kaggle input (locally it is read from `../docs/external/star_classification.csv`).

## 0. Config & imports

In [ ]:
import warnings, time, gc, os, sys
warnings.filterwarnings('ignore')
from contextlib import contextmanager
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix


@contextmanager
def quiet():
    '''Redirect C-level stdout/stderr (fd 1 & 2) to devnull for the duration
    of the block. Needed to hide native chatter that Python's warnings filter
    cannot catch — e.g. LightGBM's OpenCL kernel JIT prints "1 warning
    generated." per build when training on GPU. Our own tqdm output is written
    outside this block, so it stays visible.'''
    devnull = os.open(os.devnull, os.O_WRONLY)
    saved = [os.dup(1), os.dup(2)]
    try:
        os.dup2(devnull, 1); os.dup2(devnull, 2)
        yield
    finally:
        os.dup2(saved[0], 1); os.dup2(saved[1], 2)
        os.close(devnull); os.close(saved[0]); os.close(saved[1])

SEED = 42
N_SPLITS = 5
TARGET = 'class'
ID = 'id'
CLASSES = ['GALAXY', 'QSO', 'STAR']
class_to_int = {c: i for i, c in enumerate(CLASSES)}
int_to_class = {i: c for c, i in class_to_int.items()}

# Toggle GPU. If a booster fails on GPU in your environment, set USE_GPU = False.
USE_GPU = True
# Toggle appending the original SDSS17 dataset (add it as a Kaggle input first,
# or place star_classification.csv under ../docs/external/ for local runs).
USE_EXTERNAL = True

ON_KAGGLE = Path('/kaggle/input').exists()
if ON_KAGGLE:
    DATA_DIR = sorted(Path('/kaggle/input').rglob('train.csv'))[0].parent
    OUT_DIR = Path('/kaggle/working')
else:
    DATA_DIR = Path('../docs/dataset')
    OUT_DIR = Path('../data_processed')
print('Env:', 'Kaggle' if ON_KAGGLE else 'local', '| DATA_DIR:', DATA_DIR)

## 1. Load data

In [ ]:
def reduce_mem(df):
    for c in df.select_dtypes('float64').columns:
        df[c] = df[c].astype('float32')
    for c in df.select_dtypes('int64').columns:
        df[c] = pd.to_numeric(df[c], downcast='integer')
    return df

train = reduce_mem(pd.read_csv(DATA_DIR / 'train.csv'))
test = reduce_mem(pd.read_csv(DATA_DIR / 'test.csv'))
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')
print('train:', train.shape, '| test:', test.shape)
train.head(3)

### Optional: append original SDSS17 data
Set `USE_EXTERNAL = True` above and add the
[Stellar Classification Dataset SDSS17](https://www.kaggle.com/datasets/fedesoriano/stellar-classification-dataset-sdss17)
as a Kaggle input. We map its columns onto the competition schema and concatenate it into train.
External rows get a marker so they are excluded from OOF validation scoring.

In [ ]:
train['is_external'] = 0

if USE_EXTERNAL:
    # On Kaggle the dataset is added as an input; locally we read it from docs/external/.
    ext_paths = sorted(Path('/kaggle/input').rglob('star_classification.csv'))
    if not ext_paths:
        local_ext = DATA_DIR.parent / 'external' / 'star_classification.csv'
        if local_ext.exists():
            ext_paths = [local_ext]
    if ext_paths:
        ext = pd.read_csv(ext_paths[0])
        ext[TARGET] = ext['class'].astype(str).str.upper()
        # SDSS17 uses -9999 as a sentinel for missing photometry -> drop those rows,
        # otherwise the bands get poisoned for the boosters (EDA flagged this).
        band_ok = ~(ext[['u', 'g', 'r', 'i', 'z']] < -50).any(axis=1)
        n_bad = int((~band_ok).sum())
        ext = ext[band_ok].copy()
        # The original set lacks spectral_type / galaxy_population -> mark as missing.
        for c in ['spectral_type', 'galaxy_population']:
            if c not in ext.columns:
                ext[c] = np.nan
        common = [c for c in train.columns if c in ext.columns]
        ext = reduce_mem(ext[common].copy())
        ext['is_external'] = 1
        train = pd.concat([train, ext], ignore_index=True)
        print('Appended external rows:', len(ext), '| dropped sentinel rows:', n_bad,
              '| new train:', train.shape)
    else:
        print('External dataset not found — skipping.')

## 2. Feature engineering

In [ ]:
num_cols = ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift']
bands = ['u', 'g', 'r', 'i', 'z']
cat_cols = ['spectral_type', 'galaxy_population']

def add_features(df):
    df = df.copy()
    # All pairwise SDSS colors (band_a - band_b, a<b).
    color_cols = []
    for a in range(len(bands)):
        for b in range(a + 1, len(bands)):
            name = f'{bands[a]}_{bands[b]}'
            df[name] = df[bands[a]] - df[bands[b]]
            color_cols.append(name)
    # redshift transforms (skew taming for cleaner splits / non-tree members).
    df['redshift_log1p'] = np.log1p(df['redshift'].clip(lower=-0.999))
    # Anomaly flags motivated by EDA.
    df['is_neg_redshift'] = (df['redshift'] < 0).astype('int8')
    df['is_star_like_z'] = (df['redshift'].abs() < 0.002).astype('int8')
    # Photometry aggregates.
    df['mag_mean'] = df[bands].mean(axis=1)
    df['mag_std'] = df[bands].std(axis=1)
    return df, color_cols

train_fe, color_cols = add_features(train)
test_fe, _ = add_features(test)

extra_num = ['redshift_log1p', 'is_neg_redshift', 'is_star_like_z', 'mag_mean', 'mag_std']
print('color_cols (%d):' % len(color_cols), color_cols)
print('extra_num:', extra_num)

### Categorical encoding
- **CatBoost**: keep `spectral_type` / `galaxy_population` as native string categoricals (NaN -> "NA").
- **LightGBM / XGBoost**: ordinal integer codes (spectral type carries a physical hot->cool order).
- `redshift x spectral_type_code` interaction (top-2 signals from EDA).

In [ ]:
spectral_order = {'O/B': 0, 'A/F': 1, 'G/K': 2, 'M': 3}
pop_map = {'Blue_Cloud': 0, 'Red_Sequence': 1}

for df in (train_fe, test_fe):
    df['spectral_type_code'] = df['spectral_type'].map(spectral_order).fillna(-1).astype('int16')
    df['galaxy_population_code'] = df['galaxy_population'].map(pop_map).fillna(-1).astype('int16')
    df['z_x_spectral'] = df['redshift'] * (df['spectral_type_code'] + 1)

feature_cols = (num_cols + color_cols + extra_num +
                ['spectral_type_code', 'galaxy_population_code', 'z_x_spectral'])

for df in (train_fe, test_fe):
    for c in cat_cols:
        df[c + '_cat'] = df[c].astype('object').where(df[c].notna(), 'NA').astype(str)
cat_feature_cols = (num_cols + color_cols + extra_num +
                    ['z_x_spectral', 'spectral_type_cat', 'galaxy_population_cat'])
cb_cat_idx = [cat_feature_cols.index('spectral_type_cat'),
              cat_feature_cols.index('galaxy_population_cat')]

train_fe['target'] = train_fe[TARGET].map(class_to_int).astype('int8')
print('LGBM/XGB features (%d)' % len(feature_cols))
print('CatBoost features (%d), cat idx %s' % (len(cat_feature_cols), cb_cat_idx))
print('null check:', train_fe[feature_cols].isnull().sum().sum(),
      test_fe[feature_cols].isnull().sum().sum())

## 3. CV setup & helpers

Each booster trains with 5-fold `StratifiedKFold`, collecting **OOF** probabilities (for honest
scoring + blend/threshold tuning) and averaged **test** probabilities. Imbalance is handled with
**balanced sample weights** so each class contributes equally — aligned with balanced accuracy.
External rows (if any) stay in training but are excluded from OOF scoring.

In [ ]:
y = train_fe['target'].values
is_real = (train_fe['is_external'].values == 0)

counts = np.bincount(y[is_real], minlength=len(CLASSES))
class_w = counts.sum() / (len(CLASSES) * np.maximum(counts, 1))
sample_w = class_w[y].astype('float32')
print('class counts:', counts, '| class weights:', class_w.round(3))

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
ext_idx = np.where(~is_real)[0]
real_idx = np.where(is_real)[0]
folds = []
for tr_r, va_r in skf.split(real_idx, y[real_idx]):
    tr = np.concatenate([real_idx[tr_r], ext_idx])
    va = real_idx[va_r]
    folds.append((tr, va))
print('folds:', [(len(t), len(v)) for t, v in folds])

X = train_fe[feature_cols]
Xc = train_fe[cat_feature_cols]
Xt = test_fe[feature_cols]
Xtc = test_fe[cat_feature_cols]
n_test = len(test_fe)
NC = len(CLASSES)

## 4. LightGBM

In [ ]:
import lightgbm as lgb
print('lgbm', lgb.__version__)

lgb_params = dict(
    objective='multiclass', num_class=NC, metric='multi_logloss',
    learning_rate=0.03, num_leaves=127, max_depth=-1,
    feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
    min_child_samples=60, reg_alpha=1.0, reg_lambda=1.0,
    n_estimators=4000, random_state=SEED, n_jobs=-1, verbose=-1,
)
if USE_GPU:
    lgb_params.update(device='gpu')

def run_lgb():
    oof = np.zeros((len(train_fe), NC), dtype='float32')
    test_pred = np.zeros((n_test, NC), dtype='float32')
    bar = tqdm(folds, desc='LightGBM', unit='fold')
    for f, (tr, va) in enumerate(bar):
        m = lgb.LGBMClassifier(**lgb_params)
        with quiet():
            m.fit(X.iloc[tr], y[tr], sample_weight=sample_w[tr],
                  eval_set=[(X.iloc[va], y[va])], eval_metric='multi_logloss',
                  callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(0)])
        oof[va] = m.predict_proba(X.iloc[va])
        test_pred += m.predict_proba(Xt) / N_SPLITS
        ba = balanced_accuracy_score(y[va], oof[va].argmax(1))
        bar.set_postfix(fold=f, best_iter=m.best_iteration_, BA=f'{ba:.5f}')
        del m; gc.collect()
    return oof, test_pred

t0 = time.time()
oof_lgb, test_lgb = run_lgb()
print('LGBM OOF BA: %.5f  (%.0fs)' % (balanced_accuracy_score(y[real_idx], oof_lgb[real_idx].argmax(1)), time.time() - t0))

## 5. XGBoost

In [ ]:
import xgboost as xgb
print('xgb', xgb.__version__)

xgb_params = dict(
    objective='multi:softprob', num_class=NC, eval_metric='mlogloss',
    learning_rate=0.03, max_depth=8, subsample=0.8, colsample_bytree=0.8,
    min_child_weight=5, reg_alpha=1.0, reg_lambda=2.0,
    n_estimators=4000, random_state=SEED, n_jobs=-1,
)
if USE_GPU:
    xgb_params.update(tree_method='hist', device='cuda')
else:
    xgb_params.update(tree_method='hist')

def run_xgb():
    oof = np.zeros((len(train_fe), NC), dtype='float32')
    test_pred = np.zeros((n_test, NC), dtype='float32')
    bar = tqdm(folds, desc='XGBoost', unit='fold')
    for f, (tr, va) in enumerate(bar):
        m = xgb.XGBClassifier(**xgb_params, early_stopping_rounds=200)
        with quiet():
            m.fit(X.iloc[tr], y[tr], sample_weight=sample_w[tr],
                  eval_set=[(X.iloc[va], y[va])], verbose=False)
        oof[va] = m.predict_proba(X.iloc[va])
        test_pred += m.predict_proba(Xt) / N_SPLITS
        ba = balanced_accuracy_score(y[va], oof[va].argmax(1))
        bar.set_postfix(fold=f, best_iter=m.best_iteration, BA=f'{ba:.5f}')
        del m; gc.collect()
    return oof, test_pred

t0 = time.time()
oof_xgb, test_xgb = run_xgb()
print('XGB OOF BA: %.5f  (%.0fs)' % (balanced_accuracy_score(y[real_idx], oof_xgb[real_idx].argmax(1)), time.time() - t0))

## 6. CatBoost (native categoricals)

In [ ]:
from catboost import CatBoostClassifier, Pool
import catboost
print('catboost', catboost.__version__)

cb_params = dict(
    loss_function='MultiClass', eval_metric='TotalF1',
    learning_rate=0.05, depth=8, l2_leaf_reg=5.0,
    iterations=4000, random_seed=SEED, verbose=0,
    auto_class_weights='Balanced',
)
if USE_GPU:
    cb_params.update(task_type='GPU', devices='0')

def run_cb():
    oof = np.zeros((len(train_fe), NC), dtype='float32')
    test_pred = np.zeros((n_test, NC), dtype='float32')
    test_pool = Pool(Xtc, cat_features=cb_cat_idx)
    bar = tqdm(folds, desc='CatBoost', unit='fold')
    for f, (tr, va) in enumerate(bar):
        tr_pool = Pool(Xc.iloc[tr], y[tr], cat_features=cb_cat_idx)
        va_pool = Pool(Xc.iloc[va], y[va], cat_features=cb_cat_idx)
        m = CatBoostClassifier(**cb_params)
        with quiet():
            m.fit(tr_pool, eval_set=va_pool, early_stopping_rounds=200, use_best_model=True)
        oof[va] = m.predict_proba(va_pool)
        test_pred += m.predict_proba(test_pool) / N_SPLITS
        ba = balanced_accuracy_score(y[va], oof[va].argmax(1))
        bar.set_postfix(fold=f, best_iter=m.get_best_iteration(), BA=f'{ba:.5f}')
        del m, tr_pool, va_pool; gc.collect()
    return oof, test_pred

t0 = time.time()
oof_cb, test_cb = run_cb()
print('CatBoost OOF BA: %.5f  (%.0fs)' % (balanced_accuracy_score(y[real_idx], oof_cb[real_idx].argmax(1)), time.time() - t0))

## 7. Stacking + per-class weight tuning

Two levers, tuned on real-train OOF only:
1. **Stacking meta-model.** Instead of a coarse grid search over blend weights, fit a
   multinomial **LogisticRegression** on the stacked OOF probabilities of the three boosters
   (3 models x 3 classes = 9 features). It learns per-class, per-model combination weights —
   strictly more expressive than a single global simplex weight. Honest meta-OOF comes from
   `cross_val_predict` (no leak); the meta is then refit on the full OOF to score the test set.
2. **Per-class multipliers** on the meta probabilities before `argmax`. For balanced
   accuracy this beats plain `argmax` — it can trade a little GALAXY recall for STAR recall
   at the confusing STAR<->GALAXY boundary.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict

yv = y[real_idx]
for k, o in {'lgb': oof_lgb, 'xgb': oof_xgb, 'cb': oof_cb}.items():
    print(f'{k:4s} OOF BA: {balanced_accuracy_score(yv, o[real_idx].argmax(1)):.5f}')

# --- reference: old coarse grid blend (kept only to measure the stacking gain) ---
import itertools
grid = np.linspace(0, 1, 11)
best_w, best_grid_ba = (1/3, 1/3, 1/3), -1
for wa, wb in itertools.product(grid, grid):
    wc = 1 - wa - wb
    if wc < -1e-9:
        continue
    b = wa * oof_lgb[real_idx] + wb * oof_xgb[real_idx] + wc * oof_cb[real_idx]
    ba = balanced_accuracy_score(yv, b.argmax(1))
    if ba > best_grid_ba:
        best_grid_ba, best_w = ba, (wa, wb, max(wc, 0.0))
print('grid-blend BA (reference): %.5f  w=%s' % (best_grid_ba, tuple(round(x, 2) for x in best_w)))

# --- stacking meta-model: multinomial logistic regression on stacked probabilities ---
def stack(a, b, c, idx=slice(None)):
    return np.hstack([a[idx], b[idx], c[idx]]).astype('float64')

S_real = stack(oof_lgb, oof_xgb, oof_cb, real_idx)        # honest OOF features
S_test = stack(test_lgb, test_xgb, test_cb)               # averaged test features

# multinomial is the default for LogisticRegression in recent sklearn; class_weight
# 'balanced' keeps the meta aligned with balanced accuracy.
meta = LogisticRegression(penalty='l2', C=1.0, max_iter=2000,
                          class_weight='balanced', n_jobs=-1)
# Honest meta-OOF via nested CV (avoids meta seeing its own training rows).
meta_oof = cross_val_predict(meta, S_real, yv, cv=skf, method='predict_proba', n_jobs=-1)
print('stacking BA (meta-OOF): %.5f' % balanced_accuracy_score(yv, meta_oof.argmax(1)))

# Refit meta on the full real OOF to transform the test set.
meta.fit(S_real, yv)
blend_oof = np.zeros((len(train_fe), NC), dtype='float32')
blend_oof[real_idx] = meta_oof                            # honest values for downstream tuning
blend_test = meta.predict_proba(S_test).astype('float32')

In [ ]:
def ba_with_weights(proba, ytrue, w):
    return balanced_accuracy_score(ytrue, (proba * w).argmax(1))

w = np.ones(NC)
base = ba_with_weights(blend_oof[real_idx], yv, w)
for _ in range(40):
    improved = False
    for c in range(NC):
        for mult in np.linspace(0.5, 2.0, 31):
            wt = w.copy(); wt[c] = mult
            s = ba_with_weights(blend_oof[real_idx], yv, wt)
            if s > base + 1e-6:
                base, w, improved = s, wt, True
    if not improved:
        break
w = w / w.mean()
print('class multipliers:', dict(zip(CLASSES, w.round(3))))
print('blend BA argmax  : %.5f' % ba_with_weights(blend_oof[real_idx], yv, np.ones(NC)))
print('blend BA weighted: %.5f' % base)

## 8. Final OOF diagnostics

In [ ]:
final_oof_pred = (blend_oof[real_idx] * w).argmax(1)
print('FINAL OOF balanced accuracy: %.5f' % balanced_accuracy_score(yv, final_oof_pred))
print()
print(classification_report(yv, final_oof_pred, target_names=CLASSES, digits=4))
print('confusion matrix (rows=true):')
print(pd.DataFrame(confusion_matrix(yv, final_oof_pred), index=CLASSES, columns=CLASSES))

## 9. Submission

In [ ]:
test_pred = (blend_test * w).argmax(1)
sub = pd.DataFrame({ID: test_fe[ID].values, TARGET: [int_to_class[i] for i in test_pred]})
assert list(sub.columns) == [ID, TARGET]
assert len(sub) == len(sample_sub)
sub.to_csv(OUT_DIR / 'submission.csv', index=False)
print('Saved', (OUT_DIR / 'submission.csv').resolve(), sub.shape)
print(sub[TARGET].value_counts(normalize=True).round(4))
sub.head()